# Multi-layout world model: fixed versus multi-layout planning cost

This compares the **same frozen I-JEPA multi-layout Wall v3 world model** with the two Band 8 cost adapters from the completed DotWall layout-transfer run. It uses the four held-out layouts and the matched planning protocol: 50 seeds per layout, 100 primitive actions maximum, five-step CEM horizon, 300 candidates, 30 elites, ten iterations, and action norm cap 3.5. The only experimental arm is the adapter used to score predicted terminal image tokens. The planner selects the best CEM sequence in both arms.

Choose a GPU runtime and **Run all**. The first case is a smoke check; the full 400 episodes then resume case by case if Colab disconnects. The final cells display summary and trajectory figures inline and save them to Drive. A complete run may take hours. Set `RESUME_RUN_ID` to a previous output folder name to resume.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, signal, subprocess, sys
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Choose a GPU runtime, then Run all.'
DRIVE = Path('/content/drive/MyDrive')
RESUME_RUN_ID = ''
RUN_ID = RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = DRIVE/'datasets/curriculum_cost_adapter/wall_multi_world_adapter_planning_v1'/RUN_ID
WALL_RUN = DRIVE/'datasets/wall_matched_fixed_multi_v3'
ADAPTER_RUN = DRIVE/'datasets/curriculum_cost_adapter/dotwall_layout_transfer_v1/20261110T125759955799Z'
REPO = Path('/content/i-jepa-wm-wall-adapter-planning')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT = '842310e6584261eeda6e865ce4e7c31623367e50'
DINO_ROOT = Path('/content/dino_wm')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ARTIFACTS = Path('/content/wall-adapter-planning-artifacts')
ENCODER_DRIVE = DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
MODEL_DRIVE = WALL_RUN/'multi/world_model/best.pt'
ENCODER = ARTIFACTS/'encoder.pth.tar'
MODEL = ARTIFACTS/'multi_world_model.pt'
ENCODER_SHA = '3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
MODEL_SHA = '979e2a8f1e482e4a46de350a39d33fee94f7af1262f3d5c7e63ad3e7bd58d355'

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def shell(command, cwd=None):
    command = list(map(str, command))
    print('$', ' '.join(command), flush=True)
    with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout: print(line, end='', flush=True)
            code = process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.terminate()
            raise
    if code: raise subprocess.CalledProcessError(code, command)

print('GPU:', torch.cuda.get_device_name())
print('Output:', OUTPUT)
print('Source adapters:', ADAPTER_RUN)
print('Source world model:', MODEL_DRIVE)

## Install the pinned runner and verify inputs

In [ ]:
if not REPO.exists():
    shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip() == REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain','--untracked-files=no'],text=True).strip()
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow'])
if not DINO_ROOT.exists():
    shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip()
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(MODEL_DRIVE,MODEL,MODEL_SHA)):
    assert source.is_file(), f'Missing Drive checkpoint: {source}'
    if not target.exists() or sha256(target) != expected: shutil.copy2(source,target)
    assert sha256(target) == expected, f'Checkpoint differs: {source}'
for kind in ('fixed','multi'):
    path = ADAPTER_RUN/kind/'band_08.pt'
    assert path.is_file(), f'Missing trained adapter: {path}'
    print(kind, 'Band 8 SHA256:', sha256(path))
print('Pinned source:', SOURCE_COMMIT)

## Smoke check: one paired planning case

In [ ]:
BASE = [sys.executable,'-m','src.mylonas_ijepa_wm.wall_adapter_planning',
        '--wall-run',WALL_RUN,'--adapter-run',ADAPTER_RUN,
        '--encoder-checkpoint',ENCODER,'--world-model-checkpoint',MODEL,
        '--dino-wm-root',DINO_ROOT,'--output',OUTPUT,
        '--device','cuda','--precision','fp32']
shell(BASE+['--smoke'],cwd=REPO)
print(json.dumps(json.loads((OUTPUT/'smoke/summary.json').read_text()),indent=2))

## Full paired evaluation

This runs 50 seeds on each of four unseen layouts for both costs. Each completed episode is saved immediately; rerunning with the same `RESUME_RUN_ID` skips completed cases.

In [ ]:
shell(BASE,cwd=REPO)
summary = json.loads((OUTPUT/'full/summary.json').read_text())
print('FULL SUMMARY')
print(json.dumps(summary,indent=2))

## Success rates and actual trajectories

The first figure compares success by layout. The second shows one case per layout where the arms disagree, if available, or seed 10000 otherwise. Both PNGs are saved to Drive and displayed explicitly below.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import Image, display

full = OUTPUT/'full'
summary = json.loads((full/'summary.json').read_text())
layouts = [tuple(x) for x in ((26,22),(26,38),(38,22),(38,38))]
labels = [f'{x},{y}' for x,y in layouts]
fixed = [summary['by_layout'][key]['fixed']['success_rate'] for key in labels]
multi = [summary['by_layout'][key]['multi']['success_rate'] for key in labels]
fig,ax = plt.subplots(figsize=(8,4.5),constrained_layout=True)
x = range(len(labels))
ax.bar([i-.18 for i in x],fixed,width=.36,label='Fixed adapter',color='#6f8fb8')
ax.bar([i+.18 for i in x],multi,width=.36,label='Multi-layout adapter',color='#df8b55')
ax.set(xticks=list(x),xticklabels=labels,ylim=(0,1),xlabel='Unseen wall x, door y',ylabel='Planning success rate',
       title='One frozen multi-layout world model, two planning costs')
ax.legend()
fig.savefig(full/'success_by_layout.png',dpi=160)
plt.close(fig)
display(Image(filename=str(full/'success_by_layout.png')))

fig,axes = plt.subplots(4,2,figsize=(9,17),constrained_layout=True)
chosen=[]
for row,layout in enumerate(layouts):
    for seed in range(10000,10050):
        paths = {condition:full/condition/f'wall{layout[0]}_door{layout[1]}_seed{seed}.json'
                 for condition in ('fixed','multi')}
        records = {condition:json.loads(path.read_text()) for condition,path in paths.items()}
        if records['fixed']['success'] != records['multi']['success'] or seed == 10049:
            break
    chosen.append(dict(layout=list(layout),seed=seed,
                       fixed_success=records['fixed']['success'],multi_success=records['multi']['success']))
    wx,dy=layout
    for col,condition in enumerate(('fixed','multi')):
        ax=axes[row,col]
        record=records[condition]
        xy=record['achieved_positions']
        ax.plot([p[0] for p in xy],[p[1] for p in xy],'-o',markersize=2,lw=1.5)
        for top,bottom in ((5,dy-4),(dy+4,59)):
            ax.add_patch(Rectangle((wx-3,top),6,bottom-top,facecolor='#424242'))
        ax.scatter(*xy[0],marker='o',c='green',s=55,zorder=5)
        ax.scatter(*record['goal_position'],marker='*',c='gold',edgecolor='black',s=110,zorder=5)
        ax.set(xlim=(5,59),ylim=(59,5),aspect='equal',xlabel='x',ylabel='y',
               title=f'{condition}: wall {wx}, door {dy}, seed {seed}, success={record["success"]}')
fig.savefig(full/'example_trajectories.png',dpi=150)
plt.close(fig)
(full/'example_trajectory_cases.json').write_text(json.dumps(chosen,indent=2))
display(Image(filename=str(full/'example_trajectories.png')))
print('Summary:', full/'summary.json')
print('Figures:', full/'success_by_layout.png', full/'example_trajectories.png')
print('All episode records:', full)